# Study the use of BERT for Topic Modeling with BBC data
Based on "BERTopic: Neural topic modeling with a class-based TF-IDF procedure"
Reference: https://arxiv.org/pdf/2203.05794  

This notebook will do the following:
1. Train the BerTopic implemented in bertopic python library following on similar or same data as mentioned in section 5.1 of the paper. This notebook is for BBC:

    - b) BBC (static)

Sample processed cleaned data fromat for static files:

| Document         | Clean Text     | # word count |
|--------------|-----------|------------|
| broadband ahead join internet .... | broadband ahead join internet ...  |71      |

The notebook has following steps. The data preprocesing was done in preprocessing.ipynb and that code is separate. Here the preprocessed clean data is fed.
 - 1. Train function is using BERTopic fit_trannfrom() funtion
 - 2. Then do quantitative analysis as mentioned in section 5.3. usinf gensim python package CoherenceModel.
 - 3.Then do qualitative analysis as mentioned in section 5.3. by listing top words for any one of the topic say topic 0
 - 4. Do Visualization of topics using visulize_{functions} in BERTopic model.
 - 5. Save Model: topic_model.save("<DomainName>_bertopic_model")
 - 6. Model Inference on unseen data Here we create some unseen daat manually and see the inference. topic_model.transform(unseen_docs) will do the inference .

## 1. Setup and Dependencies

### Optional: Mount Google Drive
Uncomment and run if you want to access files from your Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Install dependencies
!pip install bertopic
!pip install umap-learn hdbscan
!pip uninstall -y numpy gensim
!pip install numpy --upgrade
!pip install gensim --upgrade
!pip install sentence-transformer

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: gensim 4.3.3
Uninstalling gensim-4.3.3:
  Successfully uninstalled gensim-4.3.3
  Using cached numpy-2.2.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.5 which is incompatible.
  Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.1 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached gen

In [3]:
processed_data_path = "/content/drive/MyDrive/DLFA_IIsc_AIML/capstone_project/data_processed/" #set data path here
model_path="/content/drive/MyDrive/DLFA_IIsc_AIML/capstone_project/model/"
#data_path = "/content/drive/MyDrive/DLFA_IIsc_AIML/capstone_project/data/"

## 2. Fit BERTopic (Training for a domain or dataset)

In [4]:
#load data BBC
import pandas as pd
bbc_file = processed_data_path + "bbc_news_mindLab_corpus.csv"
df = pd.read_csv(bbc_file)  # adjust the path
docs = df['clean_text'].dropna().tolist()  # adjust column name as needed

In [5]:
# Print first 5 cleaned documents
for i, doc in enumerate(docs[:5]):
    print(f"Document {i+1}:\n{doc}\n{'-'*40}")


Document 1:
broadband ahead join internet fast accord official figure number business connect jump report broadband connection end compare nation rank world telecom body election campaign ensure affordable high speed net access american accord report broadband increasingly popular research shopping download music watch video total number business broadband rise end compare hook broadband subscriber line technology ordinary phone line support high datum speed cable lead account line broadband phone line connection accord figure
----------------------------------------
Document 2:
plan share sale owner technology dominate index plan sell share public list market operate accord document file stock market plan raise sale observer step close public icon technology boom recently pour cold water suggestion company sell share private technically public stock start trade list equity trade money sale investor buy share private filing document share technology firm company high growth potential s

In [6]:
from bertopic import BERTopic

topic_model = BERTopic(verbose=True)

In [7]:
topics, probs = topic_model.fit_transform(docs) #return list of topic and list of topic probablities per doc

2025-05-03 11:17:17,459 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

2025-05-03 11:19:46,389 - BERTopic - Embedding - Completed ✓
2025-05-03 11:19:46,390 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-03 11:20:04,869 - BERTopic - Dimensionality - Completed ✓
2025-05-03 11:20:04,872 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-03 11:20:05,006 - BERTopic - Cluster - Completed ✓
2025-05-03 11:20:05,024 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-05-03 11:20:05,442 - BERTopic - Representation - Completed ✓


## 3. Quantitative Validation

Topic Coherence measures how semantically related the top words in each topic are. A higher score means:

Top words in a topic frequently appear together in the original documents.

The topic is more interpretable to humans.

In [8]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# Get topics as word lists
topics = topic_model.get_topics()
topic_words = [[word for word, _ in topics[topic]] for topic in topics]

# Prepare texts for coherence model (same ones used for BERTopic)
texts = [doc.split() for doc in docs]  # assumes simple tokenization

dictionary = Dictionary(texts)
coherence_model = CoherenceModel(
    topics=topic_words,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score: {coherence_score:.4f}")

Coherence Score: 0.7130


### Topic Diversity

In [9]:
# from bertopic.evaluation import evaluate_topic_coherence, evaluate_topic_diversity

# coherence = evaluate_topic_coherence(topic_model.get_topics(), docs)
# diversity = evaluate_topic_diversity(topic_model.get_topics())

# print(f"Coherence: {coherence:.4f}")
# print(f"Diversity: {diversity:.4f}")

In [11]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Step 1: Load SBERT or any word embedding model to get word vectors
s_model = SentenceTransformer('all-MiniLM-L6-v2')  # A lightweight SBERT model

# Step 2: Extract the top words for each topic
topic_words = {topic_id: [word for word, _ in topic_model.get_topic(topic_id)[:5]] for topic_id in topic_model.get_topic_info().Topic if topic_id != -1}

# Step 3: Get embeddings for all top words in each topic
topic_embeddings = {}
for topic_id, words in topic_words.items():
    topic_embeddings[topic_id] = s_model.encode(words)

# Step 4: Compute pairwise cosine similarity between topics
similarity_matrix = np.zeros((len(topic_embeddings), len(topic_embeddings)))
topic_ids = list(topic_embeddings.keys())

for i, topic_id_1 in enumerate(topic_ids):
    for j, topic_id_2 in enumerate(topic_ids):
        if i != j:
            similarity_matrix[i, j] = cosine_similarity([topic_embeddings[topic_id_1].mean(axis=0)], [topic_embeddings[topic_id_2].mean(axis=0)])[0, 0]

# Step 5: Calculate diversity (1 - average similarity)
average_similarity = similarity_matrix[np.triu_indices_from(similarity_matrix, 1)].mean()  # Only upper triangle (unique pairs)
diversity = 1 - average_similarity
print(f"Topic Diversity: {diversity:.4f}")

Topic Diversity: 0.4327


## 4. Qualitative Validation

In [12]:
topic_model.get_topic_info().head()
topic_model.get_topic(0)  # Top words for topic 0


[('election', 0.025959348174977206),
 ('party', 0.023894370021933127),
 ('government', 0.0230976386475016),
 ('labour', 0.022791455289498257),
 ('tory', 0.020448251013539198),
 ('plan', 0.01644202537335422),
 ('minister', 0.015148595797868503),
 ('leader', 0.014053874217328493),
 ('public', 0.013586557776606238),
 ('tax', 0.013192515962837228)]

In [13]:
#Representative Docs
topic_model.get_representative_docs(0)[:3]  # Inspect 3 sample docs from topic 0

['tory tax cut lift spirit finally reveal scale tory tax cut win general election earmark reduce taxi pre election message party press voter warm simple vote tory government stick labour spending plan core public service include health education increase spend defence police pension leave tax cut equivalent penny basic rate income tax money efficiency save axe bureaucracy waste civil service spending plan fill black hole leave claim rest tax cut cash cut basic rate idea float include raise tax abolish reduce tax tory party urge announce eye catch election tory leader declare aim exercise open real economic policy divide labour tory election clear choice waste tax conservative party money tax traditional tory message previously suggest labour party tax rise conservative party tax cut extension labour party big spend public service tory problem lie persuade sceptical voter big spending public service low taxi insist promise election deliver street labour claim efficiency save simply add 

## 5. Visualization

In [14]:
from IPython.display import display

display(topic_model.visualize_topics())
display(topic_model.visualize_barchart(top_n_topics=10))
display(topic_model.visualize_hierarchy())

In [ ]:
# Exclude topic -1 (outliers)
num_topics = len([topic for topic in topic_model.get_topic_info().Topic if topic != -1])
print(f"Total number of topics (excluding outliers): {num_topics}")

## 6. Save Model

In [15]:
bbc_model_file = model_path + "bbc_news_model"
topic_model.save(bbc_model_file)

2025-05-03 11:22:21,934 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


## 7. Inference or model validation with unseen data
Later, You Can Use Inference
After fit_transform(), you can use transform(new_docs) on unseen data:

In [16]:
new_topics, new_probs = topic_model.transform(["Fans around the world are downloading the new charity single released to support global disaster relief. Featuring international stars and backed by major record labels, the song has already topped charts in several countries. Music stores are reporting high demand, and digital platforms have seen a surge in streaming. Proceeds will go toward humanitarian aid, with organizers hopeful the track will raise millions. Industry experts compare the success to the original Band Aid release, which became a cultural milestone decades ago."])
print(f"Assigned topic: {new_topics[0]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-05-03 11:22:33,398 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-05-03 11:22:37,571 - BERTopic - Dimensionality - Completed ✓
2025-05-03 11:22:37,572 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-05-03 11:22:37,577 - BERTopic - Cluster - Completed ✓


Assigned topic: 2


The `topic_word` hold the tuple " top keyword that describes the topic and Relevance score.
#### Releavance Score also called importance or weigh. BERTopic, these numbers come from a class-based TF-IDF variant called c-TF-IDF. It reflects:

- How representative or important the word is for that specific topic,
compared to all other topics.

Higher numbers = more defining for the topic.

In [17]:
topic_words = topic_model.get_topic(new_topics[0])
print(f"Top words for topic {new_topics[0]}:  {topic_words}")

Top words for topic 2:  [('music', 0.053593013329781024), ('band', 0.052192914463780604), ('song', 0.046804338662231514), ('album', 0.03856058503334785), ('artist', 0.029053389935702327), ('singer', 0.026742393175562824), ('single', 0.026671330248837066), ('good', 0.026628926758628754), ('award', 0.024631270837365805), ('record', 0.02454587016069331)]


### Make a lookup where topic ID has a human readable Name
While BERTopic gives you numeric topic IDs (like 2, 5, 7), those numbers by themselves don't have meaning — they just group similar documents.

To make sense of these topics, you look at their top keywords, and based on that, you assign a human-readable name to each topic.

Python code that automatically generates a table of topic IDs, their top keywords, and a suggested name (based on top 3–5 keywords) from your trained BERTopic model.

In [18]:
# Number of top words per topic to use for naming
top_n_words = 5

# Get all topic IDs (excluding outlier -1)
topic_info = topic_model.get_topic_info()
valid_topics = topic_info[topic_info.Topic != -1].Topic.tolist()

# Build a table with topic ID, top words, and label
topic_labels = []

for topic_id in valid_topics:
    words = topic_model.get_topic(topic_id)
    top_words = [word for word, _ in words[:top_n_words]]
    label = ", ".join(top_words)
    topic_labels.append({
        "Topic ID": topic_id,
         "Topic Related To": f"{top_words[0].capitalize()}",
        "Top Words": label

    })

# Create and display a DataFrame
df_labels = pd.DataFrame(topic_labels)
print(df_labels.to_string(index=False))

 Topic ID Topic Related To                                       Top Words
        0         Election       election, party, government, labour, tory
        1             Film                  film, award, actor, star, good
        2            Music                music, band, song, album, artist
        3            Share               share, profit, firm, company, bid
        4             Seed                   seed, final, open, win, match
        5          Economy           economy, rate, growth, rise, economic
        6             Game            game, console, gamer, title, release
        7             Race                 race, indoor, win, olympic, run
        8             Club         club, manager, player, contract, season
        9            Fraud     fraud, company, executive, firm, accounting
       10         Computer       computer, robot, technology, work, laptop
       11          Referee              referee, match, player, game, club
       12           Injur

In [19]:
# Create a dictionary from the DataFrame to map topic id to s
topic_name_map = dict(zip(df_labels["Topic ID"], df_labels["Topic Related To"]))

In [20]:
# Create a list of unseen documents manually
unseen_docs = [
    "The government announced a new policy on education reform today.",
    "Manchester United won their match after a thrilling penalty shootout.",
    "Scientists discovered a new exoplanet that could support life.",
    "The singer released a new charity single to raise funds for disaster relief.",
    "A major tech company unveiled its latest AI-powered smartphone."
]

In [21]:
topics, probs = topic_model.transform(unseen_docs)

for i, doc in enumerate(unseen_docs):
    topic_id = topics[i]
    topic_name = topic_name_map.get(topic_id, "Unknown or Outlier Topic")

    print(f"\nDocument {i+1}: {doc}")
    print(f"Assigned topic: {topic_id}")
    print(f"Topic Related To: {topic_name}")
    print("Top words:", topic_model.get_topic(topics[i]))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-05-03 11:22:37,743 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-05-03 11:22:39,940 - BERTopic - Dimensionality - Completed ✓
2025-05-03 11:22:39,942 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-05-03 11:22:39,945 - BERTopic - Cluster - Completed ✓



Document 1: The government announced a new policy on education reform today.
Assigned topic: 0
Topic Related To: Election
Top words: [('election', 0.025959348174977206), ('party', 0.023894370021933127), ('government', 0.0230976386475016), ('labour', 0.022791455289498257), ('tory', 0.020448251013539198), ('plan', 0.01644202537335422), ('minister', 0.015148595797868503), ('leader', 0.014053874217328493), ('public', 0.013586557776606238), ('tax', 0.013192515962837228)]

Document 2: Manchester United won their match after a thrilling penalty shootout.
Assigned topic: -1
Topic Related To: Unknown or Outlier Topic
Top words: [('game', 0.016890423165063784), ('high', 0.012176598027449331), ('firm', 0.011977881845049318), ('technology', 0.011805933490561934), ('play', 0.011761974856897688), ('player', 0.01162636738842294), ('company', 0.011217855893917522), ('mobile', 0.009757535314288017), ('market', 0.009185579830711863), ('world', 0.008848202515248076)]

Document 3: Scientists discovered a